# Image Size Experiment Analysis

This notebook analyzes the effect of **input image resolution** on RxRx1 classification performance.

The goal is not to select the image size with the highest validation accuracy alone. Instead, the experiment evaluates the trade-off among:

- validation performance,
- generalization gap,
- training runtime,
- approximate input-compute cost,
- preservation of cellular morphological information.

The final image size selected for subsequent experiments is **384 × 384**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "results").exists() and (PROJECT_ROOT.parent / "results").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPERIMENTS_PATH = PROJECT_ROOT / "results" / "experiments.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "analysis" / "image_size"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RECOMMENDED_IMAGE_SIZE = 384
REFERENCE_IMAGE_SIZE = 256

print("Project root:", PROJECT_ROOT)
print("Experiments:", EXPERIMENTS_PATH)
print("Output:", OUTPUT_DIR)

## 1. Load experiment results

Only experiments belonging to the image-size comparison are used.

The notebook expects the main experiment table to contain fields such as:

- `experiment_group`
- `image_size`
- `final_train_acc`
- `final_val_acc`
- `best_train_acc`
- `best_val_acc`
- `best_val_loss`

Runtime fields are used when available.

In [ ]:
experiments = pd.read_csv(EXPERIMENTS_PATH)

experiments.head()

In [ ]:
required_columns = [
    "experiment_group",
    "image_size",
    "final_train_acc",
    "final_train_loss",
    "final_val_acc",
    "final_val_loss",
    "best_train_acc",
    "best_train_loss",
    "best_val_acc",
    "best_val_loss",
    "best_epoch",
]

missing = sorted(set(required_columns) - set(experiments.columns))

if missing:
    raise ValueError(f"Missing required columns: {', '.join(missing)}")

numeric_columns = [
    column for column in required_columns
    if column != "experiment_group"
]

optional_numeric_columns = [
    column for column in [
        "runtime_seconds",
        "runtime_minutes",
        "runtime_per_epoch_minutes",
    ]
    if column in experiments.columns
]

experiments[numeric_columns + optional_numeric_columns] = (
    experiments[numeric_columns + optional_numeric_columns]
    .apply(pd.to_numeric, errors="coerce")
)

experiments["experiment_group"] = experiments["experiment_group"].astype("string")

## 2. Select image-size experiments

The default filter matches experiment groups containing either `image_size` or `size`.

If your naming convention is different, modify `group_pattern` below.

In [ ]:
group_pattern = r"image_size|size"

mask = experiments["experiment_group"].str.contains(
    group_pattern,
    case=False,
    regex=True,
    na=False,
)

image_size_runs = (
    experiments.loc[mask]
    .copy()
    .sort_values("image_size")
    .reset_index(drop=True)
)

if image_size_runs.empty:
    raise ValueError(
        "No image-size experiments were found. "
        "Check experiment_group naming or modify group_pattern."
    )

image_size_runs

## 3. Derived metrics

Two types of derived metrics are useful here.

### Generalization gap

A positive accuracy gap means training performance is better than validation performance:

\[
\text{Accuracy Gap} = \text{Train Accuracy} - \text{Validation Accuracy}
\]

### Approximate resolution cost

For a square image, the number of input pixels scales approximately with:

\[
H \times W = \text{image\_size}^2
\]

Using 256 × 256 as the reference:

\[
\text{Relative Pixel Cost}
=
\left(\frac{\text{image size}}{256}\right)^2
\]

This is **not an exact FLOPs measurement**, but it is a useful compute proxy for comparing resolutions.

In [ ]:
image_size_runs["final_acc_train_val_gap"] = (
    image_size_runs["final_train_acc"]
    - image_size_runs["final_val_acc"]
)

image_size_runs["best_acc_train_val_gap"] = (
    image_size_runs["best_train_acc"]
    - image_size_runs["best_val_acc"]
)

image_size_runs["final_loss_val_train_gap"] = (
    image_size_runs["final_val_loss"]
    - image_size_runs["final_train_loss"]
)

image_size_runs["best_loss_val_train_gap"] = (
    image_size_runs["best_val_loss"]
    - image_size_runs["best_train_loss"]
)

image_size_runs["pixel_count"] = image_size_runs["image_size"] ** 2

image_size_runs["relative_pixel_cost"] = (
    image_size_runs["image_size"] / REFERENCE_IMAGE_SIZE
) ** 2

image_size_runs

## 4. Summary table

This table combines model performance and computational considerations.

The most important quantities are:

- `best_val_acc`
- `final_val_acc`
- `best_val_loss`
- `best_acc_train_val_gap`
- `runtime_per_epoch_minutes` when available
- `relative_pixel_cost`

The distance from the best observed validation accuracy is also calculated so that we can see whether a larger image size provides a practically meaningful gain.

In [ ]:
summary_columns = [
    "image_size",
    "best_val_acc",
    "final_val_acc",
    "best_val_loss",
    "best_acc_train_val_gap",
    "relative_pixel_cost",
]

for column in [
    "runtime_seconds",
    "runtime_minutes",
    "runtime_per_epoch_minutes",
]:
    if column in image_size_runs.columns:
        summary_columns.append(column)

image_size_summary = (
    image_size_runs[summary_columns]
    .copy()
    .sort_values("image_size")
    .reset_index(drop=True)
)

best_accuracy = image_size_summary["best_val_acc"].max()

image_size_summary["best_val_acc_gap_from_best"] = (
    best_accuracy - image_size_summary["best_val_acc"]
)

image_size_summary["recommended"] = (
    image_size_summary["image_size"] == RECOMMENDED_IMAGE_SIZE
)

if "runtime_per_epoch_minutes" in image_size_summary.columns:
    baseline_runtime = image_size_summary.loc[
        image_size_summary["image_size"] == REFERENCE_IMAGE_SIZE,
        "runtime_per_epoch_minutes",
    ]

    if not baseline_runtime.empty:
        image_size_summary["runtime_relative_to_256"] = (
            image_size_summary["runtime_per_epoch_minutes"]
            / baseline_runtime.iloc[0]
        )

image_size_summary

## 5. Validation performance

First, compare the best and final validation accuracy across image sizes.

In [ ]:
ordered = image_size_runs.sort_values("image_size")

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    ordered["image_size"],
    ordered["best_val_acc"],
    "o-",
    label="Best validation accuracy",
)

ax.plot(
    ordered["image_size"],
    ordered["final_val_acc"],
    "s--",
    label="Final validation accuracy",
)

selected = ordered.loc[
    ordered["image_size"] == RECOMMENDED_IMAGE_SIZE
]

if not selected.empty:
    ax.scatter(
        selected["image_size"],
        selected["best_val_acc"],
        s=140,
        facecolors="none",
        edgecolors="black",
        linewidths=1.5,
        label=f"Selected: {RECOMMENDED_IMAGE_SIZE}",
    )

ax.set_title("Validation performance by image size")
ax.set_xlabel("Image size")
ax.set_ylabel("Accuracy")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(alpha=0.25)
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

## 6. Generalization gap

A larger input resolution does not automatically imply better generalization.

The train–validation gap helps determine whether additional resolution is mainly improving training fit or is actually transferring to validation performance.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    ordered["image_size"],
    ordered["best_acc_train_val_gap"],
    "o-",
    label="Best train − validation gap",
)

ax.plot(
    ordered["image_size"],
    ordered["final_acc_train_val_gap"],
    "s--",
    label="Final train − validation gap",
)

ax.axhline(0, linewidth=1)

ax.set_title("Generalization gap by image size")
ax.set_xlabel("Image size")
ax.set_ylabel("Train − validation accuracy")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(alpha=0.25)
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

## 7. Computational cost

Increasing spatial resolution rapidly increases the amount of image data processed by the model.

For example, relative to 256 × 256:

- 256 → 1.00× pixels
- 384 → 2.25× pixels
- 512 → 4.00× pixels

The actual runtime depends on GPU, batch size, data loading, model architecture, and memory behavior, so measured runtime is preferred when available.

In [ ]:
cost_table = (
    image_size_runs[
        ["image_size", "pixel_count", "relative_pixel_cost"]
    ]
    .drop_duplicates()
    .sort_values("image_size")
    .reset_index(drop=True)
)

cost_table

In [ ]:
if "runtime_per_epoch_minutes" in ordered.columns:
    runtime_data = ordered.dropna(
        subset=["runtime_per_epoch_minutes"]
    )

    if not runtime_data.empty:
        fig, ax = plt.subplots(figsize=(8, 5))

        ax.plot(
            runtime_data["image_size"],
            runtime_data["runtime_per_epoch_minutes"],
            "o-",
        )

        ax.set_title("Runtime per epoch by image size")
        ax.set_xlabel("Image size")
        ax.set_ylabel("Runtime per epoch (minutes)")
        ax.grid(alpha=0.25)

        plt.tight_layout()
        plt.show()
    else:
        print("runtime_per_epoch_minutes exists but contains no usable values.")
else:
    print("runtime_per_epoch_minutes is not available in experiments.csv.")

## 8. Performance–compute trade-off

This plot is the most important visualization for the final decision.

The x-axis represents approximate input cost, while the y-axis represents validation performance.

A useful operating point should achieve strong validation performance without moving unnecessarily far to the right on the compute axis.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(
    ordered["relative_pixel_cost"],
    ordered["best_val_acc"],
    "o-",
)

for _, row in ordered.iterrows():
    ax.annotate(
        f"{int(row['image_size'])}",
        (
            row["relative_pixel_cost"],
            row["best_val_acc"],
        ),
        xytext=(6, 6),
        textcoords="offset points",
    )

ax.set_title("Validation accuracy vs. approximate input cost")
ax.set_xlabel(f"Relative pixel count ({REFERENCE_IMAGE_SIZE} = 1×)")
ax.set_ylabel("Best validation accuracy")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(alpha=0.25)

plt.tight_layout()
plt.show()

## 9. Compare 384 with neighboring resolutions

Because the final choice is an engineering trade-off rather than a pure leaderboard selection, it is useful to inspect the selected resolution directly against the other tested sizes.

In [ ]:
selected_row = image_size_summary.loc[
    image_size_summary["image_size"] == RECOMMENDED_IMAGE_SIZE
]

if selected_row.empty:
    raise ValueError(
        f"Image size {RECOMMENDED_IMAGE_SIZE} was not found in the experiment results."
    )

selected_best_val_acc = selected_row["best_val_acc"].iloc[0]
selected_cost = selected_row["relative_pixel_cost"].iloc[0]

comparison = image_size_summary.copy()

comparison["best_val_acc_vs_384"] = (
    comparison["best_val_acc"] - selected_best_val_acc
)

comparison["pixel_cost_vs_384"] = (
    comparison["relative_pixel_cost"] / selected_cost
)

comparison[
    [
        "image_size",
        "best_val_acc",
        "best_val_acc_vs_384",
        "relative_pixel_cost",
        "pixel_cost_vs_384",
    ]
]

## 10. Decision

### Selected image size: **384 × 384**

The final resolution is selected based on the combined consideration of **model performance, computational cost, and compatibility with the RxRx1 target task**.

A higher image resolution preserves more spatial and morphological information from the cellular images, which is potentially useful for distinguishing subtle siRNA-induced phenotypic differences. However, computational cost grows rapidly with image size. For square images, 384 × 384 already contains approximately **2.25×** as many pixels as 256 × 256, while 512 × 512 contains **4×** as many.

For the current stage of the project, image-size selection should also consider the large number of downstream experiments that still need to be performed, including normalization, augmentation, architecture comparison, and hyperparameter optimization. A substantially more expensive resolution therefore multiplies the total experimental cost.

Based on this trade-off, **384 × 384** is used for subsequent experiments. It preserves more cellular spatial information than lower-resolution inputs while keeping the computational requirement substantially below larger resolutions such as 512 × 512.

Therefore, 384 is treated as the practical operating point for the remaining baseline and optimization experiments rather than selecting the resolution solely according to the highest observed validation accuracy.

## 11. Save analysis outputs

The enriched experiment table and summary are saved so that later reports can reuse the same values without rerunning the notebook.

In [ ]:
image_size_runs.to_csv(
    OUTPUT_DIR / "image_size_runs_enriched.csv",
    index=False,
)

image_size_summary.to_csv(
    OUTPUT_DIR / "image_size_summary.csv",
    index=False,
)

selected_row.to_csv(
    OUTPUT_DIR / "recommended_image_size.csv",
    index=False,
)

conclusion = (
    "Selected image size: 384 x 384. "
    "The decision is based on the trade-off among validation performance, "
    "computational cost, preservation of cellular morphological information, "
    "and the large number of downstream RxRx1 experiments still required. "
    "384 preserves more spatial information than lower resolutions while "
    "remaining substantially cheaper than larger inputs such as 512."
)

(OUTPUT_DIR / "conclusion.txt").write_text(
    conclusion,
    encoding="utf-8",
)

print("Saved analysis outputs to:", OUTPUT_DIR)